# AutoVision ResNet50 Training on Google Colab

This notebook runs the same training flow as the local project, using the uploaded dataset folder or zip named `new_data_train_ai_mix`.

Expected dataset layout after upload:

```text
new_data_train_ai_mix/
  F1/
  HATCHBACK/
  MICRO/
  PICK_UP/
  SEDAN/
  STATION_WAGON/
  SUV/
  VAN/
```

## 1. Runtime Setup

Before running: in Colab, choose `Runtime > Change runtime type > T4 GPU` or another GPU runtime.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

REPO_URL = "https://github.com/Abd2023/AutoVision.git"
PROJECT_ROOT = Path("/content/AutoVision")
DATASET_NAME = "new_data_train_ai_mix"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "resnet50_clean_round4"
ERROR_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "error_analysis" / "resnet50_clean_round4"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATA_ROOT)

In [ ]:
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print("Current directory:", Path.cwd())

In [ ]:
%pip install -q timm matplotlib scikit-learn pillow

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA is not available. Change the Colab runtime to GPU before training.")

## 2. Upload Dataset

Upload either:

- `new_data_train_ai_mix.zip`, containing the `new_data_train_ai_mix` folder
- a zip where the class folders are directly at the zip root

The notebook will place the final dataset at `AutoVision/data/new_data_train_ai_mix`.

In [ ]:
from google.colab import files

PROJECT_CLASSES = [
    "F1",
    "HATCHBACK",
    "MICRO",
    "PICK_UP",
    "SEDAN",
    "STATION_WAGON",
    "SUV",
    "VAN",
]

def looks_like_class_root(path: Path) -> bool:
    return all((path / class_name).is_dir() for class_name in PROJECT_CLASSES)

def install_uploaded_dataset() -> None:
    if DATA_ROOT.exists() and looks_like_class_root(DATA_ROOT):
        print("Dataset already found:", DATA_ROOT)
        return

    print("Upload your dataset zip now. Expected name: new_data_train_ai_mix.zip")
    uploaded = files.upload()
    zip_files = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if not zip_files:
        raise RuntimeError("No zip file uploaded. Please upload new_data_train_ai_mix.zip")

    extract_root = Path("/content/uploaded_dataset")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_files[0], "r") as archive:
        archive.extractall(extract_root)

    candidates = [
        extract_root / DATASET_NAME,
        extract_root,
    ]
    candidates.extend(path for path in extract_root.iterdir() if path.is_dir())
    source_root = next((path for path in candidates if looks_like_class_root(path)), None)
    if source_root is None:
        raise RuntimeError("Could not find class folders inside uploaded zip.")

    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    shutil.move(str(source_root), str(DATA_ROOT))
    print("Dataset installed at:", DATA_ROOT)

install_uploaded_dataset()

In [ ]:
counts = {class_name: len([p for p in (DATA_ROOT / class_name).rglob("*") if p.is_file()]) for class_name in PROJECT_CLASSES}
print(counts)
missing = [class_name for class_name in PROJECT_CLASSES if not (DATA_ROOT / class_name).is_dir()]
if missing:
    raise RuntimeError(f"Missing class folders: {missing}")

## 3. Verify Script Version

This checks that the Colab copy of the repo has the newer script arguments used by your local workflow.

In [ ]:
train_help = subprocess.check_output(["python", "notebooks/train_resnet50.py", "--help"], text=True)
freeze_help = subprocess.check_output(["python", "notebooks/freeze_dataset.py", "--help"], text=True)

required_train_flags = ["--model-name", "--loss-type"]
required_freeze_flags = ["--dedupe-exact"]
missing_train_flags = [flag for flag in required_train_flags if flag not in train_help]
missing_freeze_flags = [flag for flag in required_freeze_flags if flag not in freeze_help]

if missing_train_flags or missing_freeze_flags:
    raise RuntimeError(
        "The cloned GitHub repo is missing required script options. "
        f"Missing train flags: {missing_train_flags}; missing freeze flags: {missing_freeze_flags}. "
        "Push/upload the latest local scripts before running training."
    )

print("Script options verified.")

## 4. Freeze Dataset

This creates `data/processed/train`, `data/processed/val`, and `data/processed/test` from `data/new_data_train_ai_mix`.

In [ ]:
!python notebooks/freeze_dataset.py --clear --raw-root data/new_data_train_ai_mix --background white --image-size 224 --max-f1 1000 --max-per-class 1000 --dedupe-exact

In [ ]:
processed_root = PROJECT_ROOT / "data" / "processed"
processed_counts = {}
for split in ["train", "val", "test"]:
    processed_counts[split] = {
        class_name: len([p for p in (processed_root / split / class_name).glob("*") if p.is_file()])
        for class_name in PROJECT_CLASSES
    }
processed_counts

## 5. Train ResNet50

This is the same command you planned locally, using the Colab GPU.

In [ ]:
!python notebooks/train_resnet50.py --data-root data/processed --output-dir notebooks/outputs/resnet50_clean_round4 --model-name resnet50 --epochs 25 --freeze-epochs 3 --batch-size 32 --device cuda --num-workers 0 --patience 8 --loss-type cross_entropy

## 6. Analyze Test Errors

In [ ]:
!python notebooks/analyze_resnet50_errors.py --data-root data/processed --checkpoint notebooks/outputs/resnet50_clean_round4/best_resnet50.pt --output-dir notebooks/outputs/error_analysis/resnet50_clean_round4 --split test --device cuda

In [ ]:
print("Test metrics:")
!cat notebooks/outputs/resnet50_clean_round4/test_metrics.json
print("\nConfusion counts:")
!cat notebooks/outputs/error_analysis/resnet50_clean_round4/confusion_counts_test.csv

## 7. Download Results

In [ ]:
from google.colab import files

download_zip = Path("/content/resnet50_clean_round4_colab_outputs.zip")
if download_zip.exists():
    download_zip.unlink()

with zipfile.ZipFile(download_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in [OUTPUT_DIR, ERROR_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(PROJECT_ROOT))

print("Created:", download_zip)
files.download(str(download_zip))